# Aşama 10: Gama ışını kaynaklarıyla çoklu haberci araştırması

Aşama 6'daki stacking'in sorunu, kaynakları **IceCube'un sonucuna bakarak** seçmemizdi. Bu aşamada kaynak listesini **nötrino verisine hiç bakmadan**, bağımsız bir gözlemden belirliyoruz: Fermi uydusunun gama ışını kataloğu (4FGL-DR4).

**Fikir:** Nötrino üreten bir kozmik hızlandırıcı, aynı süreçte gama ışını da üretir. Gökyüzünün en parlak gama ışını kaynakları, nötrino kaynağı için iyi adaylardır. Bu "çoklu haberci" (multi-messenger) astronominin temel fikri.

**Önceden kaydedilen hipotez** (veriye bakmadan önce yazıyoruz, sonradan değiştirmiyoruz):
> Kuzey gökyüzündeki (Dec > −5°) gama ışınında en parlak **100 blazardan** nötrino fazlalığı var mı?

Blazarlar, jeti doğrudan bize dönük aktif galaksi çekirdekleri. TXS 0506+056 da bir blazar.

In [ ]:
import sys; sys.path.append('../src')
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import icecube_ps as ps
importlib.reload(ps)
assert float(ps.SURUM) >= float('10.0'), f"src/icecube_ps.py eski sürüm ({getattr(ps, 'SURUM', 'yok')}). Yenisini kopyala, kernel'i yeniden başlat."
try:
    from tqdm.auto import tqdm          # ilerleme çubuğu
except ImportError:
    tqdm = None

pd.set_option('display.float_format', '{:.3g}'.format)
DATA, RESULTS = Path('../data'), Path('../results')
ev = ps.veri_yukle(DATA)
B = ps.ArkaPlan(ev['dec'].values)
print('icecube_ps', ps.SURUM, '| olay:', f'{len(ev):,}')

## 1. Fermi-LAT kataloğu

Dosya yoksa otomatik indiriliyor (~10 MB). İndirme başarısız olursa [Fermi sayfasından](https://fermi.gsfc.nasa.gov/ssc/data/access/lat/14yr_catalog/) `gll_psc_v35.fit` dosyasını elle indirip `data/` klasörüne koy.

In [ ]:
import urllib.request
fermi_dosya = DATA / 'gll_psc_v35.fit'
if not fermi_dosya.exists():
    print('Fermi kataloğu indiriliyor...')
    urllib.request.urlretrieve(
        'https://fermi.gsfc.nasa.gov/ssc/data/access/lat/14yr_catalog/gll_psc_v35.fit', fermi_dosya)
fermi = ps.fermi_katalog_yukle(fermi_dosya)
print(f'{len(fermi):,} gama ışını kaynağı')
fermi['sinif'].str.lower().value_counts().head(8)

## 2. Kaynak listesi (önceden belirlenmiş)

Sınıflar: `bll` (BL Lac), `fsrq` (düz spektrumlu kuasar), `bcu` (türü belirsiz blazar). Parlaklık ölçüsü: 100 MeV – 100 GeV arası enerji akısı.

In [ ]:
BLAZAR = {'bll', 'fsrq', 'bcu'}
K = 100
secim = fermi[fermi['sinif'].str.lower().isin(BLAZAR) & (fermi['dec'] > -5) & (fermi['dec'] < 85)]
liste = secim.sort_values('enerji_akisi', ascending=False).head(K).reset_index(drop=True)
liste.to_csv(RESULTS / 'asama10_kaynak_listesi.csv', index=False)
print(f'{len(secim)} kuzey blazarı arasından en parlak {len(liste)} tanesi seçildi.')
txs = liste['iliskili'].str.contains('TXS 0506', na=False)
print('TXS 0506+056 listede mi?', 'evet, sıra ' + str(liste.index[txs][0] + 1) if txs.any() else 'hayır')
liste[['ad', 'iliskili', 'sinif', 'ra', 'dec', 'enerji_akisi']].head(10)

In [ ]:
fig = plt.figure(figsize=(14, 7))
ax = fig.add_subplot(111, projection='mollweide')
gx = lambda r: -(np.radians(r) - np.pi)
harita = RESULTS / 'asama7_harita.npz'
if harita.exists():                       # Aşama 7'nin nötrino haritası arka planda
    h = np.load(harita)
    renk = -np.log10(ps.yerel_p(h['ts']))
    o = np.argsort(renk)
    sc = ax.scatter(gx(h['ra'][o]), np.radians(h['dec'][o]), c=renk[o], s=2, cmap='magma',
                    vmin=0, vmax=5, lw=0)
    fig.colorbar(sc, ax=ax, shrink=0.55, label='nötrino: −log₁₀(yerel p)')
boyut = 20 + 180 * (liste['enerji_akisi'] / liste['enerji_akisi'].max())
ax.scatter(gx(liste['ra']), np.radians(liste['dec']), s=boyut, facecolors='none',
           edgecolors='#4c9be8', lw=1.2, label='En parlak 100 gama ışını blazarı')
ax.set_xticks(np.radians(np.arange(-150, 151, 30)))
ax.set_xticklabels([f'{(180 - t) % 360:.0f}°' for t in np.arange(-150, 151, 30)])
ax.grid(alpha=0.3); ax.legend(loc='lower right')
ax.set_title('Nötrino gökyüzü ve gama ışını blazarları')
plt.savefig(RESULTS / 'asama10_blazar_haritasi.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. İki test

1. **Stacking:** 100 blazarın toplam sinyali (ağırlık: dedektörün o yöndeki hassasiyeti, Aşama 5).
2. **Katalog taraması:** Her blazar tek tek; en sıcak olanın anlamlılığı "100 yere baktık" düzeltmesiyle.

İkisi de aynı karıştırma döngüsünde hesaplanıyor. ⏱️ 1000 karıştırma, birkaç dakika.

In [ ]:
alanlar = ps.etkin_alanlari_yukle(DATA)
sureler = ps.canli_sureler_yukle(DATA)
agirlik = [ps.aki_basina_olay(d, 2.0, alanlar, sureler) for d in liste['dec']]
A = ps.KatalogAnalizi(ev, B, liste['ra'], liste['dec'], agirlik)
ns_st, ts_st, ts_k = A.analiz()
print(f'Stacking: nₛ = {ns_st:.1f}, TS = {ts_st:.2f}')
print(f'En sıcak tek blazar: {liste.loc[np.argmax(ts_k), "iliskili"] or liste.loc[np.argmax(ts_k), "ad"]}, TS = {ts_k.max():.2f}')

In [ ]:
N_KARISIK = 1000
dosya = RESULTS / 'asama10_karisik.npy'
kayit = list(np.load(dosya)) if dosya.exists() else []        # her satır: (stacking TS, en büyük tek TS)
rng = np.random.default_rng(1000 + len(kayit))
dongu = range(len(kayit), N_KARISIK)
for i in (tqdm(dongu, desc='karıştırma') if tqdm else dongu):
    _, t_s, t_k = A.analiz(A.karistir(rng))
    kayit.append((t_s, t_k.max()))
    if i % 50 == 49 or i == N_KARISIK - 1:
        np.save(dosya, np.array(kayit))
kayit = np.array(kayit)
print(f'{len(kayit)} karıştırma hazır')

In [ ]:
ts_st_arka, ts_max_arka = kayit[:, 0], kayit[:, 1]

# 1) Stacking
p_st_mc, p_st_w = ps.p_degeri(ts_st, ts_st_arka)
# 2) Katalog taraması: yerel (Wilks) ve 100 kaynağa bakma düzeltmesi (karıştırmalardan)
p_yerel = ps.yerel_p(ts_k.max())
p_post = (np.sum(ts_max_arka >= ts_k.max()) + 1) / (len(ts_max_arka) + 1)

sig = lambda p: max(0.0, ps.sigma(p))
print(f'Stacking (100 blazar):  {sig(p_st_mc):.2f}σ (MC) | {sig(p_st_w):.2f}σ (Wilks)')
print(f'En sıcak blazar:        yerel {sig(p_yerel):.2f}σ → 100 kaynak düzeltmesiyle {sig(p_post):.2f}σ')

liste['TS'] = ts_k
liste['yerel_sigma'] = [sig(ps.yerel_p(t)) for t in ts_k]
en_sicak = liste.sort_values('TS', ascending=False).head(5)
en_sicak.to_csv(RESULTS / 'asama10_en_sicak_blazarlar.csv', index=False)
pd.DataFrame([dict(stacking_ns=ns_st, stacking_TS=ts_st, stacking_sigma=sig(p_st_w),
                   en_sicak=en_sicak.iloc[0]['iliskili'], en_sicak_yerel_sigma=sig(p_yerel),
                   en_sicak_post_sigma=sig(p_post))]).to_csv(RESULTS / 'asama10_ozet.csv', index=False)
en_sicak[['iliskili', 'sinif', 'ra', 'dec', 'TS', 'yerel_sigma']]

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
axs[0].hist(ts_st_arka, bins=40, color='#7a8fb8', label='karıştırılmış')
axs[0].axvline(ts_st, color='#d1495b', lw=2.5, label=f'gözlenen TS = {ts_st:.1f}')
axs[0].set_yscale('log'); axs[0].set_xlabel('TS'); axs[0].legend()
axs[0].set_title(f'100 blazar stacking: {sig(p_st_w):.1f}σ')
axs[1].hist(ts_max_arka, bins=40, color='#7a8fb8', label='karıştırılmış (en sıcak)')
axs[1].axvline(ts_k.max(), color='#d1495b', lw=2.5, label=f'gözlenen en sıcak TS = {ts_k.max():.1f}')
axs[1].set_xlabel('en büyük tek kaynak TS'); axs[1].legend()
axs[1].set_title(f'Katalog taraması: düzeltilmiş {sig(p_post):.1f}σ')
plt.tight_layout()
plt.savefig(RESULTS / 'asama10_sonuclar.png', dpi=150, bbox_inches='tight')
plt.show()

## Yorumlarken

- **Bu, projedeki en "dürüst" test.** Kaynak listesini nötrino verisine bakmadan belirledik; sonuç ne çıkarsa çıksın, yorumu temiz.
- **Anlamlı bir fazlalık çıkmazsa** bu da önemli bir bilimsel sonuç: "gama ışınında en parlak blazarlar, nötrino gökyüzünün baskın kaynakları değil". IceCube da benzer analizlerle blazarların toplam astrofiziksel nötrino akısına katkısının küçük olduğunu buldu. NGC 1068'in bir blazar değil, gama ışınında görece sönük bir Seyfert galaksisi olması da bu tabloyla uyumlu: nötrino kaynakları gama ışınında en parlak kaynaklar olmak zorunda değil.
- **TXS 0506+056 listede yoksa** şaşırma: gama ışınında görece parlak ama en parlak 100 içine girmeyebilir. Önceden belirlediğimiz kuralı sonradan "TXS de olsun" diye değiştirmek tam olarak kaçındığımız şey.
- **Sadeleştirmeler:** Sadece yön (enerji terimi yok), ağırlıklarda gama ışını parlaklığını kullanmadık, kaynak sayısı (100) keyfi seçildi.